# 🇯🇵 Japan Natality Analysis (2009–2024)
### Births by mother's age group
*Source: Vital Statistics of Japan, Volume 2 Natality*

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

## 1. Data Loading & Preparation

In [ ]:
FILE_PATH = '/Users/rodrigobravo/Desktop/Jupyter/Japan_Maternity_Work_Analysis/Natality_Japan/Yearly_born/All_years.xlsx'

raw = pd.read_excel(FILE_PATH, sheet_name='Data')

age_map = {'-14 years': '<15', '50-': '50+'}
raw['Age range'] = raw['Age range'].replace(age_map)

age_groups = ['<15', '15-19', '20-24', '25-29', '30-34', '35-39', '40-44', '45-49', '50+']
filtered = raw[raw['Age range'].isin(age_groups)]

df = filtered.groupby(['Year', 'Age range'])['Born'].sum().unstack('Age range')[age_groups]
df.index.name = 'Year'

urban = filtered[filtered['Area'] == 'Urban'].groupby(['Year', 'Age range'])['Born'].sum().unstack('Age range')[age_groups]
rural = filtered[filtered['Area'] == 'Rural'].groupby(['Year', 'Age range'])['Born'].sum().unstack('Age range')[age_groups]

print(f'Data loaded from: {FILE_PATH}')
print(f'{len(df)} years  ·  {len(age_groups)} age groups')
df

## 2. Total Births Over Time

In [ ]:
totals = df.sum(axis=1)
decline = (totals.iloc[-1] - totals.iloc[0]) / totals.iloc[0] * 100

fig, ax = plt.subplots(figsize=(12, 4))
ax.fill_between(totals.index, totals.values, alpha=0.15, color='#2E75B6')
ax.plot(totals.index, totals.values, color='#2E75B6', linewidth=2.5, marker='o', markersize=5)

for year in [totals.index[0], totals.index[len(totals)//2], totals.index[-1]]:
    ax.annotate(f'{totals[year]:,.0f}',
                xy=(year, totals[year]), xytext=(0, 12), textcoords='offset points',
                ha='center', fontsize=9, fontweight='bold', color='#2E75B6')

ax.set_title(f'Total Births in Japan  (overall decline: {decline:.1f}%)', fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel('Year')
ax.set_ylabel('Births')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.2f}M' if x >= 1e6 else f'{x:,.0f}'))
ax.set_xticks(df.index)
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

print(f'{df.index[0]}: {totals.iloc[0]:,.0f} births')
print(f'{df.index[-1]}: {totals.iloc[-1]:,.0f} births')
print(f'Decline: {totals.iloc[0]-totals.iloc[-1]:,.0f} births ({decline:.1f}%)')

## 3. Births by Age Group

In [ ]:
colors = {
    '<15': '#d62728', '15-19': '#ff7f0e', '20-24': '#e377c2',
    '25-29': '#2ca02c', '30-34': '#1f77b4', '35-39': '#9467bd',
    '40-44': '#8c564b', '45-49': '#bcbd22', '50+': '#17becf',
}

fig, axes = plt.subplots(3, 3, figsize=(14, 10))
axes = axes.flatten()

for i, age in enumerate(age_groups):
    ax = axes[i]
    vals = df[age]
    pct = (vals.iloc[-1] - vals.iloc[0]) / vals.iloc[0] * 100
    color = colors[age]
    ax.fill_between(vals.index, vals.values, alpha=0.12, color=color)
    ax.plot(vals.index, vals.values, color=color, linewidth=2, marker='o', markersize=3)
    arrow = '▼' if pct < 0 else '▲'
    pct_color = '#cc0000' if pct < 0 else '#006600'
    ax.set_title(f'{age} years', fontsize=11, fontweight='bold')
    ax.text(0.97, 0.95, f'{arrow} {abs(pct):.1f}%', transform=ax.transAxes,
            ha='right', va='top', fontsize=10, fontweight='bold', color=pct_color)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k' if x >= 1000 else f'{x:.0f}'))
    ax.set_xticks(vals.index[::3])
    ax.tick_params(labelsize=8)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

fig.suptitle("Births by Mother's Age Group — Japan", fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 4. Absolute and Percentage Change

In [ ]:
abs_change = df.iloc[-1] - df.iloc[0]
pct_change = (abs_change / df.iloc[0] * 100).round(1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

bar_colors = ['#cc2222' if v < 0 else '#2a7a2a' for v in abs_change.values]
bars = ax1.barh(age_groups, abs_change.values, color=bar_colors, edgecolor='white', height=0.6)
for bar, val in zip(bars, abs_change.values):
    x = val - 2000 if val < 0 else val + 500
    ax1.text(x, bar.get_y() + bar.get_height()/2, f'{val:+,.0f}',
             va='center', ha='right' if val < 0 else 'left', fontsize=9, fontweight='bold')
ax1.axvline(0, color='black', linewidth=0.8)
ax1.set_title(f'Absolute Change\n{df.index[0]} → {df.index[-1]}', fontsize=11, fontweight='bold')
ax1.set_xlabel('Births')
ax1.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))

bar_colors2 = ['#cc2222' if v < 0 else '#2a7a2a' for v in pct_change.values]
bars2 = ax2.barh(age_groups, pct_change.values, color=bar_colors2, edgecolor='white', height=0.6)
for bar, val in zip(bars2, pct_change.values):
    x = val - 1 if val < 0 else val + 0.5
    ax2.text(x, bar.get_y() + bar.get_height()/2, f'{val:+.1f}%',
             va='center', ha='right' if val < 0 else 'left', fontsize=9, fontweight='bold')
ax2.axvline(0, color='black', linewidth=0.8)
ax2.set_title(f'Percentage Change\n{df.index[0]} → {df.index[-1]}', fontsize=11, fontweight='bold')
ax2.set_xlabel('Change %')

plt.suptitle('Impact by Age Group — Japan', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 5. Urban vs Rural

In [ ]:
pct_df = df.div(df.sum(axis=1), axis=0) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Chart 1: young age groups with first and last value
ax = axes[0]
for age, color in [('15-19', '#ff7f0e'), ('20-24', '#e377c2'), ('25-29', '#2ca02c')]:
    vals = df[age]
    ax.plot(df.index, vals, label=age, color=color, linewidth=2.5, marker='o', markersize=4)
    ax.annotate(f'{vals.iloc[0]:,.0f}',
                xy=(vals.index[0], vals.iloc[0]),
                xytext=(-8, 6), textcoords='offset points',
                fontsize=7.5, fontweight='bold', color=color, ha='right')
    ax.annotate(f'{vals.iloc[-1]:,.0f}',
                xy=(vals.index[-1], vals.iloc[-1]),
                xytext=(6, 0), textcoords='offset points',
                fontsize=7.5, fontweight='bold', color=color, ha='left')
ax.set_title('Decline in Young Age Groups (15-29)', fontsize=11, fontweight='bold')
ax.set_ylabel('Births')
ax.set_xlabel('Year')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
ax.legend(title='Age group')
ax.set_xticks(df.index)
ax.tick_params(axis='x', rotation=45)
if 2020 in df.index:
    ax.axvline(2020, color='gray', linestyle='--', alpha=0.5)
    ax.text(2020.1, ax.get_ylim()[1]*0.95, 'COVID-19', fontsize=8, color='gray')

# Chart 2: 20-24 urban vs rural with first and last value
ax2 = axes[1]
for series, label, color, marker in [
    (urban['20-24'], 'Urban', '#1f77b4', 'o'),
    (rural['20-24'], 'Rural', '#2ca02c', 's'),
]:
    ax2.plot(series.index, series.values, label=label, color=color, linewidth=2.5, marker=marker, markersize=4)
    ax2.annotate(f'{series.iloc[0]:,.0f}',
                 xy=(series.index[0], series.iloc[0]),
                 xytext=(-8, 6), textcoords='offset points',
                 fontsize=7.5, fontweight='bold', color=color, ha='right')
    ax2.annotate(f'{series.iloc[-1]:,.0f}',
                 xy=(series.index[-1], series.iloc[-1]),
                 xytext=(6, 0), textcoords='offset points',
                 fontsize=7.5, fontweight='bold', color=color, ha='left')
ax2.set_title('Age Group 20-24: Urban vs Rural', fontsize=11, fontweight='bold')
ax2.set_ylabel('Births')
ax2.set_xlabel('Year')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
ax2.legend()
ax2.set_xticks(df.index)
ax2.tick_params(axis='x', rotation=45)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.suptitle('Focus: Age Group 20-24', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 6. Focus: Age Group 20-24

In [ ]:
pct_df = df.div(df.sum(axis=1), axis=0) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for age, color in [('15-19', '#ff7f0e'), ('20-24', '#e377c2'), ('25-29', '#2ca02c')]:
    ax.plot(df.index, df[age], label=age, color=color, linewidth=2.5, marker='o', markersize=4)
ax.set_title('Decline in Young Age Groups (15-29)', fontsize=11, fontweight='bold')
ax.set_ylabel('Births')
ax.set_xlabel('Year')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
ax.legend(title='Age group')
ax.set_xticks(df.index)
ax.tick_params(axis='x', rotation=45)
if 2020 in df.index:
    ax.axvline(2020, color='gray', linestyle='--', alpha=0.5)
    ax.text(2020.1, ax.get_ylim()[1]*0.95, 'COVID-19', fontsize=8, color='gray')

ax2 = axes[1]
ax2.plot(urban.index, urban['20-24'], label='Urban', color='#1f77b4', linewidth=2.5, marker='o', markersize=4)
ax2.plot(rural.index, rural['20-24'], label='Rural', color='#2ca02c', linewidth=2.5, marker='s', markersize=4)
ax2.set_title('Age Group 20-24: Urban vs Rural', fontsize=11, fontweight='bold')
ax2.set_ylabel('Births')
ax2.set_xlabel('Year')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
ax2.legend()
ax2.set_xticks(df.index)
ax2.tick_params(axis='x', rotation=45)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.suptitle('Focus: Age Group 20-24', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 7. Age Shift: Is Motherhood Being Postponed?

In [ ]:
midpoints = {'<15': 14, '15-19': 17, '20-24': 22, '25-29': 27,
             '30-34': 32, '35-39': 37, '40-44': 42, '45-49': 47, '50+': 51}

weighted_age = pd.Series({
    year: sum(df.loc[year, age] * midpoints[age] for age in age_groups) / df.loc[year].sum()
    for year in df.index
})

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(weighted_age.index, weighted_age.values, color='#7030A0', linewidth=2.5, marker='o', markersize=5)
ax1.fill_between(weighted_age.index, weighted_age.values, weighted_age.min()-0.1, alpha=0.1, color='#7030A0')
for year in [weighted_age.index[0], weighted_age.index[-1]]:
    ax1.annotate(f'{weighted_age[year]:.2f}',
                 xy=(year, weighted_age[year]), xytext=(0, 10), textcoords='offset points',
                 ha='center', fontsize=9, fontweight='bold', color='#7030A0')
ax1.set_title("Average Mother's Age at Birth", fontsize=11, fontweight='bold')
ax1.set_ylabel('Average age (years)')
ax1.set_xlabel('Year')
ax1.set_xticks(df.index)
ax1.tick_params(axis='x', rotation=45)

ax2.plot(df.index, pct_df['20-24'], label='20-24', color='#e377c2', linewidth=2.5, marker='o', markersize=4)
ax2.plot(df.index, pct_df['30-34'], label='30-34', color='#1f77b4', linewidth=2.5, marker='o', markersize=4)
ax2.plot(df.index, pct_df['35-39'], label='35-39', color='#9467bd', linewidth=2, marker='s', markersize=4, linestyle='--')
ax2.set_title('Is Motherhood Shifting to 30+?\nRelative weight by age group', fontsize=11, fontweight='bold')
ax2.set_ylabel('% of total births')
ax2.set_xlabel('Year')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1f}%'))
ax2.legend(title='Age group')
ax2.set_xticks(df.index)
ax2.tick_params(axis='x', rotation=45)

plt.suptitle('Age Shift in Motherhood — Japan', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print(f'Average age {df.index[0]}: {weighted_age.iloc[0]:.2f} years')
print(f'Average age {df.index[-1]}: {weighted_age.iloc[-1]:.2f} years')
print(f'Shift: +{weighted_age.iloc[-1]-weighted_age.iloc[0]:.2f} years over the period')

## 8. Births Inside and Outside of Marriage by Age Group

In [ ]:
wedlock     = filtered.groupby(['Year', 'Age range'])['Born in wedlock'].sum().unstack('Age range')[age_groups]
out_wedlock = filtered.groupby(['Year', 'Age range'])['Born out of wedlock'].sum().unstack('Age range')[age_groups]

fig, axes = plt.subplots(3, 3, figsize=(15, 11))
axes = axes.flatten()

for i, age in enumerate(age_groups):
    ax = axes[i]
    w  = wedlock[age]
    ow = out_wedlock[age]
    total = w + ow

    ax.stackplot(wedlock.index,
                 [w.values, ow.values],
                 labels=['Born in wedlock', 'Born out of wedlock'],
                 colors=[colors[age], '#dddddd'],
                 alpha=0.85)

    pct_ini = ow.iloc[0] / total.iloc[0] * 100
    pct_fin = ow.iloc[-1] / total.iloc[-1] * 100
    ax.set_title(f'{age} years', fontsize=10, fontweight='bold')
    ax.text(0.97, 0.97, f'Out of wedlock: {pct_ini:.1f}% → {pct_fin:.1f}%',
            transform=ax.transAxes, ha='right', va='top',
            fontsize=7.5, color='#555555',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7))
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k' if x >= 1000 else f'{x:.0f}'))
    ax.set_xticks(wedlock.index[::3])
    ax.tick_params(labelsize=7.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

handles, labels_leg = axes[0].get_legend_handles_labels()
fig.legend(handles, labels_leg, loc='lower center', ncol=2, fontsize=10,
           bbox_to_anchor=(0.5, -0.02), frameon=False)
fig.suptitle('Births Inside and Outside of Marriage by Age Group — Japan',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 9. Evolution of Out-of-Wedlock Birth Rate by Age Group

In [ ]:
# % of births out of wedlock by age group
pct_out = (out_wedlock / (wedlock + out_wedlock) * 100).round(2)

fig, axes = plt.subplots(3, 3, figsize=(15, 11))
axes = axes.flatten()

for i, age in enumerate(age_groups):
    ax = axes[i]
    vals = pct_out[age]
    color = colors[age]

    ax.fill_between(vals.index, vals.values, alpha=0.15, color=color)
    ax.plot(vals.index, vals.values, color=color, linewidth=2.5, marker='o', markersize=4)

    ax.annotate(f'{vals.iloc[0]:.1f}%',
                xy=(vals.index[0], vals.iloc[0]),
                xytext=(6, 4), textcoords='offset points',
                fontsize=8, fontweight='bold', color=color)
    ax.annotate(f'{vals.iloc[-1]:.1f}%',
                xy=(vals.index[-1], vals.iloc[-1]),
                xytext=(-6, 4), textcoords='offset points',
                fontsize=8, fontweight='bold', color=color, ha='right')

    diff = vals.iloc[-1] - vals.iloc[0]
    arrow = '▲' if diff > 0 else '▼'
    arrow_color = '#cc0000' if diff > 0 else '#006600'
    ax.text(0.97, 0.08, f'{arrow} {abs(diff):.1f}pp',
            transform=ax.transAxes, ha='right', va='bottom',
            fontsize=8, fontweight='bold', color=arrow_color)

    ax.set_title(f'{age} years', fontsize=10, fontweight='bold')
    ax.set_ylabel('%')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1f}%'))
    ax.set_xticks(vals.index[::3])
    ax.tick_params(labelsize=7.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

fig.suptitle('Evolution of % Births Out of Wedlock by Age Group — Japan',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('\n% out of wedlock — first vs last year:')
for age in age_groups:
    diff = pct_out[age].iloc[-1] - pct_out[age].iloc[0]
    arrow = '▲' if diff > 0 else '▼'
    print(f'  {age:>6}: {pct_out[age].iloc[0]:.1f}% → {pct_out[age].iloc[-1]:.1f}%  ({arrow} {abs(diff):.1f}pp)')

## 10. Executive Summary

In [ ]:
print('=' * 60)
print(f'   SUMMARY: JAPAN NATALITY {df.index[0]}–{df.index[-1]}')
print('=' * 60)
print(f'\n📉 Total births {df.index[0]}: {totals.iloc[0]:>10,.0f}')
print(f'📉 Total births {df.index[-1]}: {totals.iloc[-1]:>10,.0f}')
print(f'   Total decline:        {totals.iloc[0]-totals.iloc[-1]:>10,.0f} ({decline:.1f}%)')

print('\n🔴 Age groups with LARGEST percentage decline:')
for age in pct_change.sort_values().head(5).index:
    print(f'   {age:>6}: {pct_change[age]:>+7.1f}%  ({abs_change[age]:>+8,.0f} births)')

print('\n🟢 Age groups on the RISE:')
rising = pct_change[pct_change > 0]
if len(rising):
    for age in rising.index:
        print(f'   {age:>6}: {pct_change[age]:>+7.1f}%  ({abs_change[age]:>+8,.0f} births)')
else:
    print('   None — all age groups declined')

print(f'\n⏩ Age shift:')
print(f'   Average age {df.index[0]}: {weighted_age.iloc[0]:.2f} years')
print(f'   Average age {df.index[-1]}: {weighted_age.iloc[-1]:.2f} years')
print(f'   Motherhood postponed by +{weighted_age.iloc[-1]-weighted_age.iloc[0]:.2f} years on average')

print('\n💡 Conclusion on 20-24 age group:')
print('   Young women are not necessarily choosing not to have')
print('   children — economic, cultural and educational pressures')
print('   are pushing the decision toward age 30+.')
print('=' * 60)